# Zimbabwe VACS 2017 — PUD exploration

Primary extract **`ZIMBABWE_VACS_2017_PUD.dta`** in **`data/raw/Zimbabwe Stata/`**. **`pyreadstat.read_dta`** loads the file; Stata **variable labels** are in **`meta`** (this extract has no embedded value-label maps in the reader).

**Harmonized geography (admin template):** **`prov`** = **Admin 1 / GeoLevel1**. **`district1`** = district **name** (layer between province and EA—not **Admin 2**). **`GeoCode`** = **enumeration area (EA)** = **Admin 2 / GeoLevel2**; the **Data User Guide** also uses it as the **survey cluster**—duplicate **GeoLevel2** and **Cluster** rows in Excel and **cross-reference in notes**. **Finer:** **`ward`**, **`sect`**. **`psu`** (numeric) is present—document vs **`GeoCode`** per the guide.

**About row IDs:** This PUD does **not** release a single respondent-level ID column. **§1** shows **raw** design / geo columns only. **§2b** adds **`PUD_ROWID`** (unique within this file order) and **`EA_KEY`** (copy of **`GeoCode`**) for joins and EDA—**not** a substitute for a documented person key if one exists in supplements.

**Flow:** §1 Load (raw ID/design) → §2 column list & EDA → **checklist** (`utils.checklist`) → **§2b** derived keys → §3 samples & slot summaries → §4 harmonized TSV.


In [4]:
from pathlib import Path
import sys

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import re

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

COUNTRY_DIR = ROOT / "data" / "raw" / "Zimbabwe Stata"
PUD_PATH = COUNTRY_DIR / "ZIMBABWE_VACS_2017_PUD.dta"
READ_KW = {}

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")


## 1. Load data

**Raw extract only:** preview table is limited to **identifying / design** columns from the file (no derived keys).


In [5]:
if not PUD_PATH.is_file():
    raise FileNotFoundError(PUD_PATH)

df, meta = pyreadstat.read_dta(PUD_PATH, **READ_KW)
print(f"File: {PUD_PATH}")
print(f"Rows × columns: {df.shape[0]:,} × {df.shape[1]:,}")
if getattr(meta, "file_label", None):
    print(f"Stata dataset label: {meta.file_label!r}")

df = df.copy()

if "sex" in df.columns:
    print("sex (confirm 1=male / 2=female in documentation):")
    display(df["sex"].value_counts(dropna=False).sort_index())

# --- Raw identifying / design columns only (no derived keys in §1) ---
print(
    f"GeoCode unique (EA level): {df['GeoCode'].nunique():,} — not one row per respondent; "
    f"total rows: {len(df):,}"
)
print(
    f"Rows per GeoCode — min {int(df.groupby('GeoCode').size().min())}, "
    f"median {df.groupby('GeoCode').size().median():.1f}, "
    f"max {int(df.groupby('GeoCode').size().max())}"
)
print(f"psu nunique: {df['psu'].nunique():,} (compare to GeoCode per User Guide)")
print(f"Duplicate rows (all columns): {int(df.duplicated().sum())}")

_h = df["hdate_vf"].dropna()
if len(_h):
    print(
        "hdate_vf (numeric YYYYMMDD) min/max:",
        int(_h.astype(np.int64).min()),
        int(_h.astype(np.int64).max()),
    )

_raw_preview = [
    "GeoCode", "district1", "prov", "ward", "sect", "psu", "finalwt", "hcluster",
    "hdate_vf", "ntot", "nsel", "sex", "h1_hh", "h1_mm", "h2", "h3",
]
assert all(c in df.columns for c in _raw_preview), _raw_preview
df[_raw_preview].head(4)


File: /Users/starsrain/research_side_projects_ipv/data/raw/Zimbabwe Stata/ZIMBABWE_VACS_2017_PUD.dta
Rows × columns: 8,715 × 670
sex (confirm 1=male / 2=female in documentation):


sex
1     803
2    7912
Name: count, dtype: int64

GeoCode unique (EA level): 1,116 — not one row per respondent; total rows: 8,715
Rows per GeoCode — min 1, median 8.0, max 21
psu nunique: 1,115 (compare to GeoCode per User Guide)
Duplicate rows (all columns): 9
hdate_vf (numeric YYYYMMDD) min/max: 20170101 20170810


,GeoCode,district1,prov,ward,sect,psu,finalwt,hcluster,hdate_vf,ntot,nsel,sex,h1_hh,h1_mm,h2,h3
0,9212350671,harare,9,23,50,1007,2455.120816,1,20170216,5,1,1,16,40,1,41
1,9212350671,harare,9,23,50,1007,2455.120816,1,20170216,7,1,1,12,49,2,54
2,9212350671,harare,9,23,50,1007,2455.120816,1,20170401,4,1,1,12,13,1,20
3,9212750240,harare,9,27,50,1017,2666.948779,1,20170402,1,1,1,10,28,1,23


## 2. Column list & quick EDA

Uses **`df` straight from §1** (only columns in the `.dta`). After **§2b**, derived columns **`PUD_ROWID`** and **`EA_KEY`** are appended.


In [6]:
name_to_label = dict(meta.column_names_to_labels) if meta.column_names_to_labels else {}

var_table = pd.DataFrame({
    "column": df.columns,
    "stata_label": [name_to_label.get(c, "") or "" for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "missing_n": df.isna().sum().values,
    "missing_pct": (100 * df.isna().mean()).round(2),
})
print(f"Variables: {len(df.columns):,}  |  Observations: {len(df):,}")
display(var_table.head(40))
display(var_table.sort_values("missing_pct", ascending=False).head(15).reset_index(drop=True))
df.info(max_cols=18)


Variables: 670  |  Observations: 8,715


,column,stata_label,dtype,missing_n,missing_pct
GeoCode,GeoCode,GeoCode,str,0,0.00
district1,district1,,str,0,0.00
finalwt,finalwt,,float64,0,0.00
prov,prov,province,int64,0,0.00
ward,ward,ward,int64,0,0.00
sect,sect,sector,int64,0,0.00
hcluster,hcluster,CLUSTER TYPE,int64,0,0.00
hdate_vf,hdate_vf,Final Visit Date,int64,0,0.00
hiv_rqf,hiv_rqf,HIV BIOMARKER QUESTIONNAIRE FINAL RESULT CODE,object,1,0.01
ntot,ntot,Number of people living in this household,int64,0,0.00


,column,stata_label,dtype,missing_n,missing_pct
0,q438_ot,438. Other (Specify),object,8715,100.00
1,q1000i,Q1000i Inconsistence Status,object,8715,100.00
2,q146_ot,146. Other (Specify),object,8715,100.00
3,q1006af,1006AF. Was this helpline Childline?,object,8715,100.00
4,q132_ot,132. Other (Specify),object,8715,100.00
5,q138_ot,138. Other (Specify),object,8715,100.00
6,q430_ot,430. Other (Specify),object,8715,100.00
7,q159af,159AF. Was this helpline Childline?,object,8715,100.00
8,q917,"917. This first time, how many people pressure...",object,8713,99.98
9,q905,"905. This last time, how many people pressured...",object,8713,99.98


<class 'pandas.DataFrame'>
RangeIndex: 8715 entries, 0 to 8714
Columns: 670 entries, GeoCode to hivpos
dtypes: float64(2), int64(58), object(452), str(158)
memory usage: 44.5+ MB


### Harmonized geography / ID checklist (you map columns yourself)

**Admin 2 = enumeration areas (EAs)** — here **`GeoCode`**, not **`district1`**. **`ward`** / **`sect`** are finer geography. After §2, **you** choose **`CANDIDATES`**; **`utils/checklist.py`** summarizes what exists in `df`.

The next cell builds **`checklist_df`** and **TSV** for Excel (**`type_and_width`**, **`suggested_layout`** for dates). See **`skills/memory.md`**.

**PI “digits”:** **character count** of the usual printed value (no zero-padding).


In [7]:
# You choose candidate columns after §2 EDA; utils only summarize what exists in `df`.
from utils.checklist import build_checklist_df, checklist_to_tsv

CANDIDATES = [
    ("Admin 1 (province)", ["prov"]),
    ("District (name; ~1.5)", ["district1"]),
    ("Admin 2 — enumeration area (EA)", ["GeoCode"]),
    ("Cluster (svy; same as GeoCode)", ["GeoCode"]),
    ("Finer geo — ward", ["ward"]),
    ("Finer geo — sect", ["sect"]),
    ("PSU (numeric)", ["psu"]),
    ("Split-sample / design (hcluster)", ["hcluster"]),
    ("Household / selection context", ["ntot", "nsel"]),
    ("Field / visit date", ["hdate_vf"]),
    ("Sex", ["sex"]),
]

_labels = meta.column_names_to_labels or {}
checklist_df = build_checklist_df(df, CANDIDATES, column_labels=_labels)

with pd.option_context("display.max_colwidth", 100, "display.width", 220):
    display(checklist_df)

print("\n--- TSV (copy for Excel / codebook) ---\n")
print(checklist_to_tsv(checklist_df))


,slot,column,stata_label,type_and_width,suggested_layout,dtype,nunique,missing_n,missing_pct,pi_digits_char_usual_display,min_nonnull,max_nonnull,sample_first_3,slot_notes
0,Admin 1 (province),prov,province,int; 1 digit,NaN,int64,10,0,0.0,1,0,9,"9, 9, 9",<NA>
1,District (name; ~1.5),district1,,str; 3–16 digits,NaN,str,90,0,0.0,3–16,<NA>,<NA>,"'harare', 'harare', 'harare'",<NA>
2,Admin 2 — enumeration area (EA),GeoCode,GeoCode,str; 10 digits,NaN,str,1116,0,0.0,10,<NA>,<NA>,"'9212350671', '9212350671', '9212350671'",<NA>
3,Cluster (svy; same as GeoCode),GeoCode,GeoCode,str; 10 digits,NaN,str,1116,0,0.0,10,<NA>,<NA>,"'9212350671', '9212350671', '9212350671'",<NA>
4,Finer geo — ward,ward,ward,int; 1–2 digits,NaN,int64,46,0,0.0,1–2,1,46,"23, 23, 23",<NA>
5,Finer geo — sect,sect,sector,int; 2 digits,NaN,int64,7,0,0.0,2,10,70,"50, 50, 50",<NA>
6,PSU (numeric),psu,,int; 1–4 digits,NaN,int64,1115,0,0.0,1–4,1,1118,"1007, 1007, 1007",<NA>
7,Split-sample / design (hcluster),hcluster,CLUSTER TYPE,int; 1 digit,NaN,int64,2,0,0.0,1,1,2,"1, 1, 1",<NA>
8,Household / selection context,ntot,Number of people living in this household,int; 1–2 digits,NaN,int64,21,0,0.0,1–2,1,26,"5, 7, 4",<NA>
9,Household / selection context,nsel,Number of ER in this household,int; 1 digit,NaN,int64,6,0,0.0,1,1,6,"1, 1, 1",<NA>



--- TSV (copy for Excel / codebook) ---

slot	column	stata_label	type_and_width	suggested_layout	dtype	nunique	missing_n	missing_pct	pi_digits_char_usual_display	min_nonnull	max_nonnull	sample_first_3	slot_notes
Admin 1 (province)	prov	province	int; 1 digit		int64	10	0	0.0	1	0	9	9, 9, 9	
District (name; ~1.5)	district1		str; 3–16 digits		str	90	0	0.0	3–16			'harare', 'harare', 'harare'	
Admin 2 — enumeration area (EA)	GeoCode	GeoCode	str; 10 digits		str	1116	0	0.0	10			'9212350671', '9212350671', '9212350671'	
Cluster (svy; same as GeoCode)	GeoCode	GeoCode	str; 10 digits		str	1116	0	0.0	10			'9212350671', '9212350671', '9212350671'	
Finer geo — ward	ward	ward	int; 1–2 digits		int64	46	0	0.0	1–2	1	46	23, 23, 23	
Finer geo — sect	sect	sector	int; 2 digits		int64	7	0	0.0	2	10	70	50, 50, 50	
PSU (numeric)	psu		int; 1–4 digits		int64	1115	0	0.0	1–4	1	1118	1007, 1007, 1007	
Split-sample / design (hcluster)	hcluster	CLUSTER TYPE	int; 1 digit		int64	2	0	0.0	1	1	2	1, 1, 1	
Household / selectio

## 2b. Derived row keys (for EDA below)

These fields are **not** in the Stata file as released; they help **row-level tracking** and **EA-level joins**. **`GeoCode`** remains the raw EA identifier.


In [8]:
# No single released respondent ID — stable row index + explicit EA alias.
df["PUD_ROWID"] = np.arange(len(df), dtype=np.int64)
df["EA_KEY"] = df["GeoCode"].astype(str)

_dup_all = int(df.duplicated().sum())
print(
    f"Derived PUD_ROWID (0..n-1, unique per row in this file) and EA_KEY (= GeoCode str).\n"
    f"Exact duplicate rows (all columns): {_dup_all} — inspect if you need a natural person key from supplements."
)
print(f"GeoCode nunique (EA): {df['GeoCode'].nunique():,} | psu nunique: {df['psu'].nunique():,}")


Derived PUD_ROWID (0..n-1, unique per row in this file) and EA_KEY (= GeoCode str).
Exact duplicate rows (all columns): 0 — inspect if you need a natural person key from supplements.
GeoCode nunique (EA): 1,116 | psu nunique: 1,115


## 3. Further EDA and exploration

Run **§2b** first so **`PUD_ROWID`** / **`EA_KEY`** exist.

### Raw row samples


In [9]:
pd.set_option("display.max_columns", 42)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 72)

_core = [c for c in [
    "PUD_ROWID", "EA_KEY", "GeoCode", "district1", "prov", "ward", "sect", "psu",
    "finalwt", "hcluster", "hdate_vf", "ntot", "nsel", "sex", "h2", "h3", "q2", "asr",
] if c in df.columns]
display(df[_core].head(8))
display(df[_core].sample(6, random_state=0))


,PUD_ROWID,EA_KEY,GeoCode,district1,prov,ward,sect,psu,finalwt,hcluster,hdate_vf,ntot,nsel,sex,h2,h3,q2,asr
0,0,9212350671,9212350671,harare,9,23,50,1007,2455.120816,1,20170216,5,1,1,1,41,22,22
1,1,9212350671,9212350671,harare,9,23,50,1007,2455.120816,1,20170216,7,1,1,2,54,14,14
2,2,9212350671,9212350671,harare,9,23,50,1007,2455.120816,1,20170401,4,1,1,1,20,20,20
3,3,9212750240,9212750240,harare,9,27,50,1017,2666.948779,1,20170402,1,1,1,1,23,23,23
4,4,9212750240,9212750240,harare,9,27,50,1017,2666.948779,1,20170403,1,1,1,1,22,22,22
5,5,9212750240,9212750240,harare,9,27,50,1017,2666.948779,1,20170403,3,1,1,1,23,23,23
6,6,9212750240,9212750240,harare,9,27,50,1017,2666.948779,1,20170408,7,2,1,1,37,14,14
7,7,9212750240,9212750240,harare,9,27,50,1017,2666.948779,1,20170408,4,1,1,1,52,17,17


,PUD_ROWID,EA_KEY,GeoCode,district1,prov,ward,sect,psu,finalwt,hcluster,hdate_vf,ntot,nsel,sex,h2,h3,q2,asr
5486,5486,4210850040,4210850040,chinhoyi,4,8,50,589,208.170378,2,20170517,8,1,2,1,22,22,22
4162,4162,2211250200,2211250200,bindura urban,2,12,50,423,284.896402,2,20170507,6,1,2,1,34,13,13
841,841,9212350125,9212350125,harare,9,23,50,1004,220.098889,2,20170216,4,1,2,NaN,NaN,20,20
2971,2971,1070910090,1070910090,nyanga,1,9,10,274,77.531060,2,20170726,4,1,2,1,22,22,22
6678,6678,7032210120,7032210120,gokwe south,7,22,10,739,286.298916,2,20170624,9,1,2,2,59,19,19
8576,8576,9211550050,9211550050,harare,9,15,50,981,262.573191,2,20170216,4,1,2,NaN,NaN,22,22


### Slot summaries (ID / geo / design)


In [10]:
L = meta.column_names_to_labels or {}

_WORD_SEX = re.compile(r"\b(?:male|females?|female)\b", re.IGNORECASE)
_EMBED_MF = re.compile(r"(?i)(?:^|_)(?:male|female)(?=_|$)")


def _abstract_digit_pattern(val: str) -> str:
    parts = []
    i = 0
    while i < len(val):
        ch = val[i]
        if ch.isdigit():
            j = i
            while j < len(val) and val[j].isdigit():
                j += 1
            parts.append("N" * (j - i))
            i = j
        elif ch.isalpha():
            j = i
            while j < len(val) and val[j].isalpha():
                j += 1
            parts.append("A")
            i = j
        else:
            parts.append(ch)
            i += 1
    return "".join(parts)


def _unified_style_pattern(st: pd.Series):
    st = st.dropna().astype(str)
    if len(st) == 0:
        return None
    abstracts = st.map(_abstract_digit_pattern)
    if abstracts.nunique(dropna=False) != 1:
        return None
    pat = abstracts.iloc[0]
    if not any(ch.isdigit() for ch in pat):
        return None
    if set(pat) <= {"N"}:
        return None
    ex = st.iloc[0]
    if len(pat) > 72:
        return f"{pat[:72]}… (e.g. {ex[:40]}{'…' if len(ex) > 40 else ''})"
    return f"{pat} (e.g. {ex})"


def _width_note(s: pd.Series) -> str:
    sn = s.dropna()
    if len(sn) == 0:
        return "n/a"
    if pd.api.types.is_numeric_dtype(s):
        whole = (sn == sn.astype(float).astype(int)).all()
        if whole:
            lens = sn.astype(int).astype(str).str.len()
            lo, hi = int(lens.min()), int(lens.max())
            return f"{lo}-{hi} digits (integer codes)" if lo != hi else f"{lo} digits (integer codes)"
        lens = sn.astype(str).str.len()
        lo, hi = int(lens.min()), int(lens.max())
        return f"{lo}-{hi} chars (numeric as string)" if lo != hi else f"{lo} chars (numeric as string)"
    st = sn.astype(str)
    lens = st.str.len()
    lo, hi = int(lens.min()), int(lens.max())
    w = f"{lo}-{hi} chars" if lo != hi else f"{lo} chars"
    if st.str.fullmatch(r"\d+").all():
        return f"{w} (string; all numeric characters)"
    return f"{w} (string)"


def _special_id_note(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s):
        return "no M/F identifier (numeric)"
    st = s.dropna().astype(str)
    if len(st) == 0:
        return "n/a"
    if st.str.contains(_WORD_SEX, regex=True, na=False).any() or st.str.contains(_EMBED_MF, regex=True, na=False).any():
        return "Male/Female text (words or _Female_/_Male_ segments)"
    return "no male/female text (heuristic)"


def slot_summary(title: str, cols: list, note_extra: str = ""):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print("MISSING columns:", miss)
        return
    for c in cols:
        s = df[c]
        lbl = (str(L.get(c) or ""))[:75]
        size_part = _width_note(s)
        style = _unified_style_pattern(s) if not pd.api.types.is_numeric_dtype(s) else None
        style_part = f"; style {style}" if style else ""
        id_part = _special_id_note(s)
        if pd.api.types.is_numeric_dtype(s):
            sn = s.dropna()
            extra = f"min/max={sn.min()}/{sn.max()}" if len(sn) else "min/max=n/a"
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}; {extra}"
        else:
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}"
        print(f"  {c} | {lbl}")
        print(f"    {info}; n_distinct={s.nunique(dropna=True)}; missing={s.isna().sum()}")
    if note_extra:
        print("  ", note_extra)


slot_summary(
    "1. Derived row keys",
    ["PUD_ROWID", "EA_KEY"],
    "PUD_ROWID = row order in this file; EA_KEY = str(GeoCode) for EA joins",
)
slot_summary(
    "2. GeoCode alone (EA — not respondent-unique)",
    ["GeoCode"],
    "**Admin 2** / cluster id; many adolescents per EA",
)
slot_summary("3. psu (numeric)", ["psu"], "Compare mapping to GeoCode in User Guide")
slot_summary("4. Province & district name", ["prov", "district1"], "prov = GeoLevel1; district1 not Admin 2")
slot_summary(
    "5. Geography stack",
    ["GeoCode", "district1", "prov", "ward", "sect", "psu"],
    "**GeoLevel2** = GeoCode; district1 = name layer; ward/sect finer",
)
slot_summary("6. hcluster (design)", ["hcluster"], "Split-sample / cluster type; cross-check vs sex")
slot_summary("7. Weights", ["finalwt"], "Confirm pweight / `svyset` in User Guide")
slot_summary("8. Final visit date", ["hdate_vf"], "Numeric YYYYMMDD in this extract")
slot_summary("9. Interview clock (HoH / start)", ["h1_hh", "h1_mm"], "")
slot_summary("10. Roster / selection fields", ["ntot", "nsel"], "Within-EA structure—not a unique person key by themselves")



1. Derived row keys
  PUD_ROWID | 
    1-4 digits (integer codes); no M/F identifier (numeric); dtype=int64; min/max=0/8714; n_distinct=8715; missing=0
  EA_KEY | 
    10 chars (string; all numeric characters); no male/female text (heuristic); dtype=str; n_distinct=1116; missing=0
   PUD_ROWID = row order in this file; EA_KEY = str(GeoCode) for EA joins

2. GeoCode alone (EA — not respondent-unique)
  GeoCode | GeoCode
    10 chars (string; all numeric characters); no male/female text (heuristic); dtype=str; n_distinct=1116; missing=0
   **Admin 2** / cluster id; many adolescents per EA

3. psu (numeric)
  psu | 
    1-4 digits (integer codes); no M/F identifier (numeric); dtype=int64; min/max=1/1118; n_distinct=1115; missing=0
   Compare mapping to GeoCode in User Guide

4. Province & district name
  prov | province
    1 digits (integer codes); no M/F identifier (numeric); dtype=int64; min/max=0/9; n_distinct=10; missing=0
  district1 | 
    3-16 chars (string); no male/female text 

## 4. Harmonized codebook slots (Zimbabwe 2017)

**Single PUD.** **`GeoCode`** is both **GeoLevel2 (EA)** and **Cluster** for survey variance—use two Excel rows or one row with rich **notes**.

### Row ID — **no released person key in this extract**

- **`PUD_ROWID`**: unique **within this file** (row order as read from `.dta`); use for EDA joins to this extract only.
- **`EA_KEY` / `GeoCode`**: **~1,116** EAs; **not** one row per respondent.
- Check **User Guide** / supplements if a documented respondent identifier exists elsewhere.

```
slot	variable_male	variable_female	type_and_width	notes
Row pointer (derived)	(derive) PUD_ROWID	same	int	Unique **within this `.dta` read order**; not a published survey ID
EA alias (derived)	(derive) EA_KEY (=GeoCode)	same	str; 10 digits	Convenience for EA-level merges; same as GeoCode
EA id (not respondent-unique)	GeoCode	same	str; 10 digits	**Admin 2**; User Guide **cluster**
Cluster (svy)	GeoCode	same	str; 10 digits	Same variable as GeoLevel2—cross-reference in notes
Geo level 1 (province)	prov	same	int	Province code; label province
District (name)	district1	same	str; 3–16 chars	**Not** Admin 2; lowercase names in extract
Finer geo	ward sect	ward sect			Sub-EA; confirm in guide
PSU (numeric)	psu	same			Relate to GeoCode per User Guide
Split-sample	hcluster	same			Design flag; cross-check vs sex
Analysis weight	finalwt	same	float	Confirm `svy` syntax in guide
Roster / selection context	ntot nsel	ntot nsel			Within-EA structure; not a unique person key alone
Final visit date	hdate_vf	same	int; 8 digits; YYYYMMDD	See checklist suggested_layout
Interview clock	h1_hh h1_mm	h1_hh h1_mm			Head-of-household / interview start fields
Sex	sex	same			Confirm coding in questionnaire
```
